In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# Cài đặt các phiên bản thư viện đã được kiểm chứng an toàn cho Kaggle
!pip install -q "sentence-transformers<3.3.0" "transformers<4.48.0" pandas torch

# Khởi tạo môi trường sạch cho CUDA
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Cài đặt thư viện hoàn tất! Sẵn sàng xử lý dữ liệu.")

In [ ]:
import json
import os

# Định nghĩa tên file đầu vào và đầu ra
LAW_FILE = '/kaggle/input/datasets/ltanh64/dataset-after-soft-clean/law.json'
QUESTIONS_FILE = '/kaggle/input/datasets/ltanh64/dataset-after-soft-clean/questions_clean.json'

# KIỂM TRA ĐƯỜNG DẪN: 
# Nếu bạn chạy lần đầu chưa kịp up file, Cell này sẽ tự tạo file mẫu cấu trúc chuẩn của bạn để chạy test không bị crash code.
if not os.path.exists(LAW_FILE) or not os.path.exists(QUESTIONS_FILE):
    print("⚠️ Không tìm thấy file dữ liệu gốc trong thư mục làm việc. Tự động khởi tạo dữ liệu mẫu chuẩn của Nhóm 4...")
    
    with open(LAW_FILE, 'w', encoding='utf-8') as f:
        json.dump(mock_law, f, ensure_ascii=False, indent=4)
    with open(QUESTIONS_FILE, 'w', encoding='utf-8') as f:
        json.dump(mock_questions, f, ensure_ascii=False, indent=4)
        
print("✅ Xác nhận: Tệp 'law.json' và 'questions.json' đã sẵn sàng xử lý!")

In [ ]:
import json
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# ==========================================
# 1. TẢI ĐỮ LIỆU VÀO BỘ NHỚ
# ==========================================
print("Đang đọc dữ liệu vào hệ thống...")
with open('/kaggle/input/datasets/ltanh64/dataset-after-soft-clean/law.json', 'r', encoding='utf-8') as f:
    laws = json.load(f)
with open('/kaggle/input/datasets/ltanh64/dataset-after-soft-clean/questions_clean.json', 'r', encoding='utf-8') as f:
    questions = json.load(f)

# Cấu trúc từ điển tăng tốc độ tra cứu ID luật
law_dict = {item['chunk_id']: item['text_to_embed'] for item in laws}
law_ids = list(law_dict.keys())
law_texts = list(law_dict.values())

# ==========================================
# 2. KHỞI TẠO MÔ HÌNH EMBEDDING TRÊN GPU
# ==========================================
print("Đang tải mô hình ngôn ngữ BGE-M3 (Chạy trên nền CUDA)...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('BAAI/bge-m3', device=device)

# ==========================================
# 3. VECTOR HÓA TOÀN BỘ DATASET (EMBEDDING)
# ==========================================
print("Tiến hành mã hóa không gian vector cho Văn bản Luật...")
law_embeddings = model.encode(law_texts, convert_to_tensor=True, show_progress_bar=True)

print("Tiến hành mã hóa không gian vector cho Tập câu hỏi...")
q_texts = [q['question'] for q in questions]
q_embeddings = model.encode(q_texts, convert_to_tensor=True, show_progress_bar=True)

# Tính toán ma trận Cosine tương đồng hàng loạt trên GPU
print("Đang tính toán ma trận tương đồng ngữ nghĩa...")
cosine_scores = util.cos_sim(q_embeddings, law_embeddings)

# ==========================================
# 4. THUẬT TOÁN PHÂN TÁCH VÀ BẮT LỖI GÁN NHÃN
# ==========================================
clean_questions = []
noisy_questions = []

# Cấu hình ngưỡng lọc khoa học (Hyperparameters)
LOW_SCORE_THRESHOLD = 0.35  # Ngưỡng điểm tối thiểu của nhãn gốc
MARGIN_THRESHOLD = 0.15     # Khoảng cách tối thiểu so với nhãn tốt nhất thực tế

print("Thuật toán đang thực hiện quét lỗi tự động...")
for i, q in enumerate(questions):
    assigned_id = q['article_id'][0] 
    
    # Xử lý trường hợp ID gán không có trong file luật
    if assigned_id not in law_dict:
        q['error'] = "Chunk_id này hoàn toàn không tồn tại trong file law.json"
        noisy_questions.append(q)
        continue

    # Lấy điểm số hiện tại của cặp Câu hỏi - Luật được gán
    assigned_idx = law_ids.index(assigned_id)
    assigned_score = cosine_scores[i][assigned_idx].item()

    # Quét tìm xem có Điều luật nào khác khớp hơn không
    top_score = torch.max(cosine_scores[i]).item()
    top_idx = torch.argmax(cosine_scores[i]).item()
    best_matching_id = law_ids[top_idx]

    # Điều kiện bắt nhãn nhiễu (Noisy Label Rule)
    if assigned_score < LOW_SCORE_THRESHOLD and (top_score - assigned_score) > MARGIN_THRESHOLD:
        q['diagnostic'] = {
            'assigned_article': assigned_id,
            'assigned_score': round(assigned_score, 4),
            'suggested_article': best_matching_id,
            'suggested_score': round(top_score, 4),
            'suggested_text': law_dict[best_matching_id][:150] + "..."
        }
        noisy_questions.append(q)
    else:
        clean_questions.append(q)

# ==========================================
# 5. XUẤT FILE VÀ BÁO CÁO THỐNG KÊ
# ==========================================
print("\n" + "="*40)
print("📊 BÁO CÁO KẾT QUẢ ĐÃ LỌC TỰ ĐỘNG:")
print(f"  - Tổng số lượng mẫu đầu vào  : {len(questions)}")
print(f"  - Số câu hỏi đạt chuẩn (SẠCH) : {len(clean_questions)}")
print(f"  - Số câu hỏi gán sai (NHIỄU)  : {len(noisy_questions)}")
print("="*40)

# Ghi dữ liệu sạch ra file mới để nộp bài/fine-tune
with open('vector_questions_clean.json', 'w', encoding='utf-8') as f:
    json.dump(clean_questions, f, ensure_ascii=False, indent=4)

# Ghi dữ liệu nhiễu để nhóm làm báo cáo phân tích lỗi (Error Analysis)
with open('vector_questions_noisy.json', 'w', encoding='utf-8') as f:
    json.dump(noisy_questions, f, ensure_ascii=False, indent=4)

print("🎉 Hoàn tất thành công! Đã xuất các tệp 'vector_questions_clean.json' và 'vector_questions_noisy.json' ra thư mục Output.")